<a id="entropy"></a>

$$
\text{信息熵 (Entropy): 衡量一组标签的 "混乱程度" / 不确定性} \\ \quad \\
\text{1. 信息量 (Information Content): 概率为 p 的事件发生时带来的信息 } I(p) \\
\text{要求: 越罕见信息越多 (I 递减), 必然事件没有信息 } I(1) = 0 \text{, 两个独立事件的信息可加 } I(pq) = I(p) + I(q) \\
\text{满足 } I(pq) = I(p) + I(q) \text{ 的连续函数只有对数, 所以 } I(p) = -\log_2 p \text{ (以 2 为底, 单位是 bit)} \\ \quad \\
\text{2. 熵 = 信息量的期望: K 个类别的比例 } p_1, \cdots, p_K \\
H = E[I] = -\sum_{k=1}^K p_k \log_2 p_k \quad (\text{约定 } 0 \log 0 = 0 \text{, 因为 } \lim_{p \to 0} p \ln p = 0) \\ \quad \\
\text{3. 最大值 (拉格朗日乘子法): } \max H \ \text{s.t.} \ \sum_k p_k = 1 \\
L = -\sum_k p_k \ln p_k - \lambda \left(\sum_k p_k - 1\right), \quad \frac{\partial L}{\partial p_k} = -\ln p_k - 1 - \lambda = 0 \implies p_k = e^{-1 - \lambda} \\
\text{所有 } p_k \text{ 相等} \implies p_k = \frac{1}{K}, \ H_{\max} = \log_2 K \text{ (均匀分布最混乱); 只有一个类别时 } H = 0 \text{ (纯节点)} \\ \quad \\
\text{二分类: } H(p) = -p \log_2 p - (1 - p) \log_2 (1 - p) \text{, p = 0.5 时最大 = 1 bit}
$$

数学基础: [拉格朗日乘子法](../math_basics/calculus.ipynb#lagrange) · [数学期望](../math_basics/statistics.ipynb#expectation) · 对比: [交叉熵损失](logistic_regression.ipynb#cross-entropy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy
from sklearn.tree import DecisionTreeClassifier

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]                       # 0 log 0 = 0
    return -np.sum(p * np.log2(p)) + 0.0          # + 0.0 把 -0.0 变成 0.0

def gini(p):
    p = np.asarray(p, dtype=float)
    return 1 - np.sum(p**2)

for p in [[1, 0, 0], [0.5, 0.5, 0], [0.7, 0.2, 0.1], [1 / 3, 1 / 3, 1 / 3]]:
    print(f'p = {np.round(p, 3)}: H = {entropy(p):.4f} (scipy = {scipy_entropy(p, base=2):.4f}), Gini = {gini(p):.4f}')
print(f'K = 3 时熵的最大值 log2(3) = {np.log2(3):.4f}')

# 随机生成很多个 3 类分布, 熵都不超过 log2(3)
rng = np.random.default_rng(0)
random_p = rng.dirichlet(np.ones(3), 20000)
print(f'20000 个随机分布的最大熵 = {max(entropy(p) for p in random_p):.4f}')

p_grid = np.linspace(0.001, 0.999, 300)
plt.figure(figsize=(4, 3))
plt.plot(p_grid, [entropy([p, 1 - p]) for p in p_grid], label='entropy (bit)')
plt.plot(p_grid, [gini([p, 1 - p]) for p in p_grid], label='Gini')
plt.plot(p_grid, 1 - np.maximum(p_grid, 1 - p_grid), label='misclassification')
plt.xlabel('p (binary)')
plot_nicely()
plt.show()

<a id="information-gain"></a>

$$
\text{信息增益 (Information Gain) 与基尼不纯度 (Gini Impurity)} \\ \quad \\
\text{一次划分把父节点 (n 个样本) 分成左右两个子节点 (} n_L, n_R \text{ 个样本):} \\
IG = H(\text{parent}) - \left[ \frac{n_L}{n} H(L) + \frac{n_R}{n} H(R) \right] \\
\text{划分后的熵按样本数加权平均, 信息增益 = 不确定性减少了多少, 选 IG 最大的划分 (ID3 / C4.5 的做法)} \\ \quad \\
\text{基尼不纯度 (CART 的默认): } G = \sum_{k=1}^K p_k (1 - p_k) = 1 - \sum_k p_k^2 \\
\text{推导含义: 从节点里随机抽一个样本 (类别为 k 的概率 } p_k \text{), 再按节点的类别比例随机给它贴一个标签} \\
P(\text{贴错}) = \sum_k P(\text{真实为 k}) \, P(\text{标签不是 k}) = \sum_k p_k (1 - p_k) \\
\text{也就是每个类别的指示变量 (伯努利) 的方差 } p_k (1 - p_k) \text{ 之和} \\ \quad \\
\text{为什么不直接用误分类率 } 1 - \max_k p_k \text{: 它不是严格凹的, 很多划分的误分类率完全不变, 但其实让节点变得更纯了}
$$

数学基础: [伯努利分布的方差 p(1 - p)](../math_basics/statistics.ipynb#binomial)

In [ ]:
def class_probs(y, n_classes):
    return np.bincount(y, minlength=n_classes) / len(y)

# 例子: 父节点 [40 个类别 0, 40 个类别 1]
y_parent = np.array([0] * 40 + [1] * 40)
split_a = (np.array([0] * 30 + [1] * 10), np.array([0] * 10 + [1] * 30))  # 划分 A: 左 [30, 10], 右 [10, 30]
split_b = (np.array([0] * 20 + [1] * 40), np.array([0] * 20))             # 划分 B: 左 [20, 40], 右 [20, 0] (纯)
for name, (L, R) in [('A', split_a), ('B', split_b)]:
    n_p = len(y_parent)
    ig = entropy(class_probs(y_parent, 2)) - (len(L) / n_p * entropy(class_probs(L, 2)) + len(R) / n_p * entropy(class_probs(R, 2)))
    gg = gini(class_probs(y_parent, 2)) - (len(L) / n_p * gini(class_probs(L, 2)) + len(R) / n_p * gini(class_probs(R, 2)))
    miss = 0.5 - (len(L) / n_p * (1 - class_probs(L, 2).max()) + len(R) / n_p * (1 - class_probs(R, 2).max()))
    print(f'划分 {name}: 信息增益 = {ig:.4f}, 基尼下降 = {gg:.4f}, 误分类率下降 = {miss:.4f}')
print('两种划分的误分类率下降一样 (都是 0.25), 但熵和基尼都认为 B 更好 (产生了一个完全纯的节点, 不用再继续切)')

# 基尼 = 随机贴标签的错误率 (模拟)
p_node = np.array([0.6, 0.3, 0.1])
true = rng.choice(3, size=200000, p=p_node)
guess = rng.choice(3, size=200000, p=p_node)
print(f'Gini = {gini(p_node):.4f}, 模拟随机贴标签的错误率 = {np.mean(true != guess):.4f}')

<a id="best-split"></a>

$$
\text{寻找最佳划分 (连续特征)} \\
\text{对每个特征 j: 把样本按 } x_j \text{ 排序, 候选阈值取相邻两个不同取值的中点 } t = \frac{x_{(i)} + x_{(i+1)}}{2} \\
\text{规则 } x_j \leq t \text{ 去左边, 否则去右边; 计算每个候选的加权熵, 取最小 (信息增益最大) 的 (j, t)} \\ \quad \\
\text{高效计算: 排好序后, 阈值从左往右移动时, 左边的各类别计数就是标签 one-hot 的累加和 (cumsum)} \\
\text{右边计数 = 总计数 - 左边计数, 一次扫描就能得到所有阈值的不纯度, 复杂度 } O(n \log n) \text{ (排序)}
$$

In [ ]:
def entropy_rows(counts):
    # counts: m x K, 每一行是一个节点的类别计数, 返回每行的熵
    totals = counts.sum(axis=1, keepdims=True)
    p = np.divide(counts, totals, out=np.zeros_like(counts, dtype=float), where=totals > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        logs = np.where(p > 0, np.log2(p), 0.0)
    return -(p * logs).sum(axis=1)

def best_split(X, y, n_classes):
    n_s, d_s = X.shape
    parent_H = entropy(class_probs(y, n_classes))
    best = (None, None, -np.inf)                            # (特征, 阈值, 信息增益)
    for j in range(d_s):
        order = np.argsort(X[:, j], kind='stable')
        xs, ys = X[order, j], y[order]
        left = np.cumsum(np.eye(n_classes)[ys], axis=0)[:-1]   # 前 i 个样本去左边, i = 1..n-1
        right = left[-1] + np.eye(n_classes)[ys[-1]] - left
        n_left = np.arange(1, n_s)
        child_H = (n_left * entropy_rows(left) + (n_s - n_left) * entropy_rows(right)) / n_s
        valid = xs[1:] > xs[:-1]                             # 相同取值之间不能切
        if not valid.any():
            continue
        gains = np.where(valid, parent_H - child_H, -np.inf)
        i = np.argmax(gains)
        if gains[i] > best[2]:
            best = (j, (xs[i] + xs[i + 1]) / 2, gains[i])
    return best

# 3 类的二维连续数据
rng = np.random.default_rng(1)
centers = np.array([[0, 0], [2.5, 1.0], [1.0, 2.8]])
X = np.vstack([rng.normal(c, 0.9, size=(100, 2)) for c in centers])
y = np.repeat(np.arange(3), 100)
perm = rng.permutation(len(y))
X, y = X[perm], y[perm]
X_train, y_train, X_test, y_test = X[:200], y[:200], X[200:], y[200:]

j0, t0, g0 = best_split(X_train, y_train, 3)
sk1 = DecisionTreeClassifier(criterion='entropy', max_depth=1, random_state=0).fit(X_train, y_train)
print(f'手算根节点划分: 特征 x{j0 + 1} <= {t0:.6f}, 信息增益 = {g0:.6f}')
print(f'sklearn 根节点: 特征 x{sk1.tree_.feature[0] + 1} <= {sk1.tree_.threshold[0]:.6f}')

# 暴力验证: 直接对每个阈值算一遍
xs_sorted = np.sort(X_train[:, j0])
cands = (xs_sorted[1:] + xs_sorted[:-1]) / 2
def ig_at(t):
    L, R = y_train[X_train[:, j0] <= t], y_train[X_train[:, j0] > t]
    return entropy(class_probs(y_train, 3)) - (len(L) * entropy(class_probs(L, 3)) + len(R) * entropy(class_probs(R, 3))) / len(y_train)
print(f'暴力搜索的最佳阈值 = {cands[np.argmax([ig_at(t) for t in cands])]:.6f}')

<a id="build-tree"></a>

$$
\text{递归建树 (Recursive Partitioning)} \\
\text{build(node):} \\
\quad \text{如果节点是纯的 (H = 0), 或者达到最大深度, 或者样本少于 2 个 → 叶子节点, 预测为多数类} \\
\quad \text{否则: 找最佳划分 (j, t), 递归地建左子树 } \{x_j \leq t\} \text{ 和右子树 } \{x_j > t\} \\ \quad \\
\text{这是贪心算法 (Greedy): 每一步只选当前最好的划分, 不保证整棵树全局最优 (全局最优是 NP-hard)} \\ \quad \\
\text{过拟合: 不限制深度时, 树会一直切到每个叶子都是纯的, 训练集准确率 100\%, 但把噪声也记住了} \\
\text{控制方法: 最大深度, 叶子最少样本数, 剪枝 (Pruning), 或者用很多树做集成 (随机森林, 梯度提升)}
$$

In [ ]:
def build_tree(X, y, n_classes, depth, max_depth):
    counts = np.bincount(y, minlength=n_classes)
    node = {'predict': int(np.argmax(counts)), 'counts': counts}
    if counts.max() == len(y) or depth == max_depth or len(y) < 2:
        return node
    j, t, gain = best_split(X, y, n_classes)
    if j is None:
        return node
    mask = X[:, j] <= t
    node.update(feature=j, threshold=t,
                left=build_tree(X[mask], y[mask], n_classes, depth + 1, max_depth),
                right=build_tree(X[~mask], y[~mask], n_classes, depth + 1, max_depth))
    return node

def predict_one(node, x):
    while 'feature' in node:
        node = node['left'] if x[node['feature']] <= node['threshold'] else node['right']
    return node['predict']

def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])

def print_tree(node, indent=''):
    if 'feature' not in node:
        print(f'{indent}叶子: 预测 {node["predict"]}, 计数 {node["counts"]}')
        return
    print(f'{indent}x{node["feature"] + 1} <= {node["threshold"]:.4f}?')
    print_tree(node['left'], indent + '    ')
    print_tree(node['right'], indent + '    ')

tree3 = build_tree(X_train, y_train, 3, 0, max_depth=3)
print_tree(tree3)
sk3 = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=0).fit(X_train, y_train)
g1, g2 = np.meshgrid(np.linspace(-3, 5, 150), np.linspace(-3, 5.5, 150))
grid = np.column_stack([g1.ravel(), g2.ravel()])
print(f'深度 3: 训练集预测和 sklearn 一致? {np.array_equal(predict(tree3, X_train), sk3.predict(X_train))}, '
      f'网格上一致? {np.array_equal(predict(tree3, grid), sk3.predict(grid))}')

# 不同深度: 训练准确率 vs 测试准确率
depths = range(1, 13)
train_acc, test_acc, sk_test_acc = [], [], []
for dep in depths:
    tr = build_tree(X_train, y_train, 3, 0, dep)
    train_acc.append(np.mean(predict(tr, X_train) == y_train))
    test_acc.append(np.mean(predict(tr, X_test) == y_test))
    sk_test_acc.append(DecisionTreeClassifier(criterion='entropy', max_depth=dep, random_state=0).fit(X_train, y_train).score(X_test, y_test))
print(f'训练准确率 = {np.round(train_acc, 3)}')
print(f'测试准确率 = {np.round(test_acc, 3)}')
print(f'sklearn 测试准确率 = {np.round(sk_test_acc, 3)}')
# 深度很大时, 小节点里常常有多个划分的信息增益完全相同, 手算 (按特征顺序) 和 sklearn (随机特征顺序) 选的可能不同, 结果会有微小差异

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].contourf(g1, g2, predict(tree3, grid).reshape(g1.shape), levels=[-0.5, 0.5, 1.5, 2.5], colors=['C0', 'C1', 'C2'], alpha=0.25)
for k in range(3):
    axs[0].scatter(*X_train[y_train == k].T, s=6, color=f'C{k}', label=f'class {k}')
axs[0].set_title('depth 3: axis-aligned splits')
axs[1].plot(list(depths), train_acc, 'o-', label='train')
axs[1].plot(list(depths), test_acc, 's-', label='test')
axs[1].set_xlabel('max depth')
axs[1].set_ylabel('accuracy')
plot_nicely()
plt.show()